In [21]:
import os
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch

from ippy import operators, solvers, metrics

BASE_DATA_PATH = "../data"

SINOGRAM_CLEAN_PATH = os.path.join(BASE_DATA_PATH, "sinogram_clean")
SINOGRAM_CORRUPTED_PATH = os.path.join(BASE_DATA_PATH, "sinogram_corrupted")

reso = 256

ANGLE_CONFIGS = [180, 90, 60, 45]

lmbda = 0.01  # regularization parameter
maxiter = 100  # Number of maximum iterations for the solver
p = 1  # Sparsity parameter 

batch_size = 1 #for variational method

In [14]:
class SinogramDataset(Dataset):

    def __init__(self, clean_paths, corrupted_paths, image_paths):

        self.clean_paths = sorted(clean_paths)
        self.corrupted_paths = sorted(corrupted_paths)
        self.image_paths = sorted(image_paths)

        assert len(self.clean_paths) == len(self.corrupted_paths)
        assert len(self.clean_paths) == len(self.image_paths)

    def __len__(self):
        return len(self.clean_paths)

    def __getitem__(self, idx):

        clean = np.load(self.clean_paths[idx]).astype(np.float32)
        corrupted = np.load(self.corrupted_paths[idx]).astype(np.float32)
        x_true = np.load(self.image_paths[idx]).astype(np.float32)

        # sinograms: [1, det, angles]
        clean = torch.from_numpy(clean).unsqueeze(0)
        corrupted = torch.from_numpy(corrupted).unsqueeze(0)

        # image: [1, H, W]
        x_true = torch.from_numpy(x_true).unsqueeze(0)

        return corrupted, clean, x_true

In [15]:
datasets = {
    "train": {},
    "validation": {},
    "test": {}
}

dataloaders = {
    "train": {},
    "validation": {},
    "test": {}
}



for split in ["train", "validation", "test"]:

    for n_angles in ANGLE_CONFIGS:

        image_paths = glob(
            os.path.join(
                BASE_DATA_PATH,
                "preprocessed",
                split,
                "*.npy"
            )
        )

        clean_paths = glob(
            os.path.join(
                SINOGRAM_CLEAN_PATH,
                split,
                f"angles_{n_angles}",
                "*.npy"
            )
        )

        corrupted_paths = glob(
            os.path.join(
                SINOGRAM_CORRUPTED_PATH,
                split,
                f"angles_{n_angles}",
                "*.npy"
            )
        )

        dataset = SinogramDataset(
            clean_paths,
            corrupted_paths,
            image_paths
)

        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=(split == "train")
        )

        datasets[split][n_angles] = dataset
        dataloaders[split][n_angles] = loader

operators_dict = {}

for n_angles in ANGLE_CONFIGS:

    K = operators.CTProjector(
        img_shape=(reso, reso),
        angles=np.linspace(0, np.pi, n_angles, endpoint=False),
        det_size=reso,
        geometry="parallel",
    )

    operators_dict[n_angles] = K

CUDA not available. CTProjector will use CPU.
Attempting to create ASTRA projector type: 'linear' for 'parallel' geometry...
Successfully created ASTRA projector type: 'linear'
CTProjector initialized. Geometry: parallel. Using GPU: False. FBP Algorithm: FBP
CUDA not available. CTProjector will use CPU.
Attempting to create ASTRA projector type: 'linear' for 'parallel' geometry...
Successfully created ASTRA projector type: 'linear'
CTProjector initialized. Geometry: parallel. Using GPU: False. FBP Algorithm: FBP
CUDA not available. CTProjector will use CPU.
Attempting to create ASTRA projector type: 'linear' for 'parallel' geometry...
Successfully created ASTRA projector type: 'linear'
CTProjector initialized. Geometry: parallel. Using GPU: False. FBP Algorithm: FBP
CUDA not available. CTProjector will use CPU.
Attempting to create ASTRA projector type: 'linear' for 'parallel' geometry...
Successfully created ASTRA projector type: 'linear'
CTProjector initialized. Geometry: parallel. U

/home/extter/sparse_mayo/notebooks/ippy/operators.py:134: UserWarning: CuPy not found. GPU acceleration for ASTRA via CuPy will be disabled.
  warnings.warn(


In [26]:
lambdas = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2]

results = {}

for n_angles in ANGLE_CONFIGS:

    print(f"\n===== {n_angles} ANGLES =====")

    K = operators_dict[n_angles]
    solver = solvers.ChambollePockTpVUnconstrained(K)

    loader = dataloaders["train"][n_angles]

    # take 10 samples only
    subset = []
    for i, data in enumerate(loader):
        subset.append(data)
        if i == 10:
            break

    results[n_angles] = {}

    for lmbda in lambdas:

        print(f"\n--- lambda = {lmbda} ---")

        metrics_list = []

        for idx, (y_delta, y_clean, x_true) in enumerate(subset):
            print(f"\n i in subset of 10 is:   {idx} " )
            x_sol, info = solver(
                y_delta,
                lmbda=lmbda,
                starting_point=None,
                x_true=x_true,
                maxiter=maxiter,
                p=p,
                verbose=False,
            )

            re = float(metrics.RE(x_sol, x_true))
            psnr = float(metrics.PSNR(x_sol, x_true))
            ssim = float(metrics.SSIM(x_sol, x_true))

            metrics_list.append((re, psnr, ssim))

        metrics_list = np.array(metrics_list)

        results[n_angles][lmbda] = {
            "RE": metrics_list[:,0].mean(),
            "PSNR": metrics_list[:,1].mean(),
            "SSIM": metrics_list[:,2].mean(),
        }

        print(
            f"λ={lmbda} | "
            f"RE={metrics_list[:,0].mean():.4f}, "
            f"PSNR={metrics_list[:,1].mean():.4f}, "
            f"SSIM={metrics_list[:,2].mean():.4f}"
        )


===== 180 ANGLES =====

--- lambda = 0.0001 ---

 i in subset of 10 is:   0 

 i in subset of 10 is:   1 

 i in subset of 10 is:   2 

 i in subset of 10 is:   3 

 i in subset of 10 is:   4 

 i in subset of 10 is:   5 

 i in subset of 10 is:   6 

 i in subset of 10 is:   7 

 i in subset of 10 is:   8 

 i in subset of 10 is:   9 

 i in subset of 10 is:   10 
λ=0.0001 | RE=0.1248, PSNR=30.1626, SSIM=0.7504

--- lambda = 0.0005 ---

 i in subset of 10 is:   0 

 i in subset of 10 is:   1 

 i in subset of 10 is:   2 

 i in subset of 10 is:   3 

 i in subset of 10 is:   4 

 i in subset of 10 is:   5 

 i in subset of 10 is:   6 

 i in subset of 10 is:   7 

 i in subset of 10 is:   8 

 i in subset of 10 is:   9 

 i in subset of 10 is:   10 
λ=0.0005 | RE=0.1171, PSNR=30.7245, SSIM=0.7869

--- lambda = 0.001 ---

 i in subset of 10 is:   0 

 i in subset of 10 is:   1 

 i in subset of 10 is:   2 

 i in subset of 10 is:   3 

 i in subset of 10 is:   4 

 i in subset of 10 i

In [29]:
import pandas as pd

data = {}

for n_angles in ANGLE_CONFIGS:

    data[(n_angles, "PSNR")] = [
        results[n_angles][lmbda]["PSNR"]
        for lmbda in lambdas
    ]

    data[(n_angles, "RE")] = [
        results[n_angles][lmbda]["RE"]
        for lmbda in lambdas
    ]

    data[(n_angles, "SSIM")] = [
        results[n_angles][lmbda]["SSIM"]
        for lmbda in lambdas
    ]

table = pd.DataFrame(data, index=lambdas)

table

180                            90                       \
             PSNR        RE      SSIM       PSNR        RE      SSIM   
0.0001  30.162614  0.124820  0.750373  31.422492  0.116881  0.809452   
0.0005  30.724521  0.117099  0.786864  31.519165  0.115623  0.818768   
0.0010  30.974121  0.113851  0.803073  31.484600  0.116111  0.820547   
0.0050  31.616183  0.106020  0.837749  31.642232  0.114218  0.836930   
0.0100  31.835431  0.103486  0.845661  31.759820  0.112809  0.843801   
0.0500  31.543281  0.107119  0.845854  31.305321  0.119094  0.832656   

              60                             45                       
             PSNR        RE      SSIM       PSNR        RE      SSIM  
0.0001  30.787958  0.112696  0.792812  30.748106  0.116549  0.804042  
0.0005  30.878486  0.111505  0.800227  30.865228  0.114952  0.808909  
0.0010  30.837376  0.112029  0.800574  30.839644  0.115303  0.807995  
0.0050  30.801505  0.112583  0.809453  30.689208  0.117515  0.804603  
0.0100  30.828533  0.112341  0.814313  30.589572  0.119049  0.801202  
0.0500  30.364810  0.118790  0.795648  29.856803  0.130033  0.767336